# Day 3 — Agent Memory

---

Agents that forget everything between conversations are frustrating. A good agent has **memory**:

- **Short-term** — what happened in this conversation (the message history)
- **Long-term** — what happened in past conversations (stored somewhere durable)

Today you'll build both, using the **exact vector store from Section 5** as long-term memory. No new tech — just wiring things together.

*(You may hear terms like "episodic" and "semantic" memory in blog posts. They're academic labels; you don't need them yet. Short-term + long-term is enough to build production agents.)*


## 1. Short-term memory — just a list

Short-term memory *is* the messages list you've been passing to the LLM. The LLM sees everything in there.

**Problems that appear as it grows:**
- Cost — every token costs money on every call.
- Context window — Llama-3.3-70B has 128k tokens but many models have 4k–32k.
- Quality — models get distracted by long histories (Day 4 Section 6 covered this).

**Fixes (increasing complexity):**
1. **Truncate** — keep the last N turns.
2. **Summarize** — periodically ask the LLM to compress old turns into a paragraph.
3. **Selective recall** — pull only relevant past turns into context (this is where long-term memory comes in).


In [ ]:
!pip install together sentence-transformers chromadb python-dotenv --quiet

In [4]:
class ChatMemory:
    """Sliding-window short-term memory."""

    def __init__(self, max_turns: int = 10):
        self.max_turns = max_turns
        self.messages: list[dict] = []

    def add(self, role: str, content: str) -> None:
        self.messages.append({"role": role, "content": content})
        # keep only the most recent 2*max_turns (user + assistant per turn)
        if len(self.messages) > self.max_turns * 2:
            self.messages = self.messages[-self.max_turns * 2:]

    def as_prompt(self, system: str) -> list[dict]:
        return [{"role": "system", "content": system}] + self.messages


mem = ChatMemory(max_turns=3)
mem.add("user", "Hi, I'm Uday.")
mem.add("assistant", "Hello Uday!")
mem.add("user", "What's my name?")
print(mem.as_prompt("You are helpful."))


[{'role': 'system', 'content': 'You are helpful.'}, {'role': 'user', 'content': "Hi, I'm Uday."}, {'role': 'assistant', 'content': 'Hello Uday!'}, {'role': 'user', 'content': "What's my name?"}]


**Simple, and enough for most chat apps.** The moment you need "remember what the user told you 3 weeks ago" — you need long-term memory.


## 2. Long-term memory — reuse Section 5

The trick: **write every interesting thing to a vector store**, then **retrieve relevant memories before answering.** It's RAG applied to conversation history.

Store: `(user_id, timestamp, text)` triples.
Retrieve: at every turn, search the store for memories similar to the current user message.

This is *literally* RAG. The KB is now conversation memory instead of documents.


In [2]:
from sentence_transformers import SentenceTransformer
import chromadb, os, time

embed = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.Client()
long_mem = client.create_collection("long_term")


def remember(user_id: str, text: str) -> None:
    long_mem.add(
        documents=[text],
        embeddings=embed.encode([text]).tolist(),
        metadatas=[{"user_id": user_id, "ts": time.time()}],
        ids=[f"{user_id}_{time.time_ns()}"],
    )


def recall(user_id: str, query: str, top_k: int = 3) -> list[str]:
    r = long_mem.query(
        query_embeddings=embed.encode([query]).tolist(),
        n_results=top_k,
        where={"user_id": user_id},
    )
    return r["documents"][0] if r["documents"] else []


# Seed some memories
for fact in [
    "Uday is a Python developer based in Bangalore.",
    "Uday's favorite framework is FastAPI.",
    "Uday is building an AI course.",
    "Uday hates JavaScript.",
]:
    remember("uday", fact)

print(recall("uday", "what does Uday do for work?"))
print(recall("uday", "does Uday like frontend?"))


['Uday is a Python developer based in Bangalore.', 'Uday is building an AI course.', "Uday's favorite framework is FastAPI."]
["Uday's favorite framework is FastAPI.", 'Uday hates JavaScript.', 'Uday is a Python developer based in Bangalore.']


## 3. Agent with both memories


In [3]:
from together import Together
from dotenv import load_dotenv
load_dotenv()
llm = Together()

class Chatbot:
    def __init__(self, user_id: str):
        self.user_id = user_id
        self.short = ChatMemory(max_turns=5)

    def talk(self, message: str) -> str:
        # 1. Recall relevant long-term memories
        memories = recall(self.user_id, message, top_k=3)
        mem_block = "\n".join(f"- {m}" for m in memories) or "(none)"

        system = (f"You are a helpful assistant. Below are things you remember "
                  f"about the user:\n{mem_block}\n\nUse them if relevant.")

        # 2. Add the user turn to short-term
        self.short.add("user", message)

        # 3. Build final prompt: system(with memories) + recent turns
        messages = self.short.as_prompt(system)

        # 4. Call LLM
        resp = llm.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=messages,
            temperature=0.3,
        )
        answer = resp.choices[0].message.content.strip()

        # 5. Store the turn in short-term memory
        self.short.add("assistant", answer)

        # 6. Optionally save something to long-term memory too
        # (skipped for brevity - see Day 5 assignment)

        return answer


bot = Chatbot("uday")
print(bot.talk("Should I try building a frontend for my course?"))
print("---")
print(bot.talk("What do you know about my job?"))


### Short answer  
**Yes, if you want a polished, interactive learning experience that’s accessible to everyone.**  
But if you’re just getting the course out the door, you can start with a minimal front‑end (or even a static site) and add more polish later.

---

## Why a front‑end can help

| Benefit | How it shows up in your course |
|---------|--------------------------------|
| **User‑friendly navigation** | Students can jump between modules, view progress, and see a clear roadmap. |
| **Interactive demos** | Embed live code editors, visualizations, or AI demos that run in the browser. |
| **Branding & engagement** | A custom UI can make your course feel like a product, not just a PDF. |
| **Analytics** | Track clicks, time spent, quiz scores, etc., to iterate on content. |
| **Accessibility** | Proper HTML/CSS/ARIA makes the course usable for screen readers, keyboard navigation, etc. |

---

## How to decide

| Question | What to do |
|----------|------------|
| **Do I need real‑

**What just happened on turn 2:** the user asked about their job. `recall` found the "Uday is a Python developer in Bangalore" memory (semantic similarity to "job"), added it to the system prompt, and the answer used it — even though the current conversation never mentioned Uday's job.

That's persistent memory in 40 lines.


## 4. What to actually remember (the design question)

You can't dump every message into long-term memory — signal-to-noise gets bad. Two common patterns:

**a) Explicit "remember this" facts.**
User: *"By the way, my name is Uday and I prefer Python."*
Extract those facts (with a small LLM call) and save. This is what ChatGPT's memory feature does.

**b) Summaries of past sessions.**
After each session, ask the LLM: *"Summarize what the user shared in this conversation in ≤3 bullet points."* Store the bullets, not the raw transcript.

For your first agent: **manual `remember(...)` calls when the user reveals something factual.** Auto-extraction can wait.


## 5. A useful mental model

| Kind | Where it lives | What's in it | Query cost |
|---|---|---|---|
| **Short-term** | The `messages` list in memory | Last N turns of *this* conversation | Free |
| **Long-term** | ChromaDB (or any vector store) | Facts/summaries across all past sessions | Vector search + tokens for retrieved text |
| **Reference material** | Same vector store, different collection | Docs, product info, KB | Same as RAG |

Notice long-term memory and RAG are *structurally the same*. That's why you didn't need any new library today.


## Recap

- **Short-term memory** = a sliding window of recent turns.
- **Long-term memory** = a vector store of past facts / summaries per user.
- The pattern is **RAG on your conversation history**. Same tools as Section 5.
- Store what's meaningful (explicit facts, session summaries), not raw noise.
- **Next class:** reliability — retries, human-in-the-loop, cost budgets. The unglamorous stuff that makes agents shippable.
